In [9]:
import pandas as pd

path = "../data/domains_annotated.csv"
df = pd.read_csv(path)
df.head(10)

,column A,count,category,Product Listing Page,About Page,notes
0,manifestationmatters.com,180,standalone_domain,https://manifestationmatters.com/manifestation...,https://manifestationmatters.com/category/law-...,"has shop, long essays, no direct about page, b..."
1,nextlevelsoul.com,151,standalone_domain,https://watch.nextlevelsoul.com/category/event...,https://nextlevelsoul.com/blog/,"store is appearal, but there is a courses page"
2,elliotoracle.com,89,standalone_domain,https://elliotoracle.com/shop/,https://elliotoracle.com/,home page acts as about
3,dailydevotionalinchrist.com,64,other,NaN,NaN,not related: mainly links to social media
4,icandosomething.com,39,other,NaN,NaN,weird
5,conjuredcardea.com,39,standalone_domain,https://conjuredcardea.com/collections/all-pro...,https://conjuredcardea.com/,NaN
6,bodymindspiritworld.pro,32,other,NaN,NaN,dead link
7,imberraven.com,28,standalone_domain,https://imberraven.com/collections/all,NaN,NaN
8,goodinthehead.com,28,standalone_domain,https://goodinthehead.com/books-for-sale/,https://goodinthehead.com/the-mental-hospital/,NaN
9,enchanteddragonflyma.com,28,standalone_domain,https://www.enchanteddragonflyma.com/s/shop,https://www.enchanteddragonflyma.com/#AImbkY,NaN


In [3]:
df = df[df["category"] == "standalone_domain"]
df = df.dropna(subset=["About Page"], axis=0)
df.head(10)

,column A,count,category,Product Listing Page,About Page,notes
0,manifestationmatters.com,180,standalone_domain,https://manifestationmatters.com/manifestation...,https://manifestationmatters.com/category/law-...,"has shop, long essays, no direct about page, b..."
1,nextlevelsoul.com,151,standalone_domain,https://watch.nextlevelsoul.com/category/event...,https://nextlevelsoul.com/blog/,"store is appearal, but there is a courses page"
2,elliotoracle.com,89,standalone_domain,https://elliotoracle.com/shop/,https://elliotoracle.com/,home page acts as about
5,conjuredcardea.com,39,standalone_domain,https://conjuredcardea.com/collections/all-pro...,https://conjuredcardea.com/,NaN
8,goodinthehead.com,28,standalone_domain,https://goodinthehead.com/books-for-sale/,https://goodinthehead.com/the-mental-hospital/,NaN
9,enchanteddragonflyma.com,28,standalone_domain,https://www.enchanteddragonflyma.com/s/shop,https://www.enchanteddragonflyma.com/#AImbkY,NaN
10,tanyafengshui.com,27,standalone_domain,https://tanyafengshui.com/feng-shui-products/,https://tanyafengshui.com/about-me/,NaN
12,atlantisscalar.com,26,standalone_domain,https://www.atlantisscalar.com/scalar-services/,https://www.atlantisscalar.com/about-atlantis-...,NaN
13,thesecret.tv,24,standalone_domain,https://www.thesecret.tv/store/,https://www.thesecret.tv/our-mission/,NaN
14,dramitjain.org,23,standalone_domain,https://dramitjain.org/,https://dramitjain.org/ourstory/,NaN


In [14]:
import requests
from bs4 import BeautifulSoup

def scrape_html(link):
    try:
        res = requests.get(link).text
        return BeautifulSoup(res, "html.parser")
    except: 
        return None

def extract_tags(link):
    res_html = scrape_html(link)
    return set([tag.name for tag in res_html.find_all()])

def find_content_container(soup):
    for tag in ['main', 'article']:
        el = soup.find(tag)
        if el and len(el.get_text(strip=True)) > 200:
            return el
        
    sections = soup.find_all('section')
    if sections:
        return max(sections, key=lambda s: len(s.get_text(strip=True)))
    
    divs = soup.find_all('div')
    if divs:
        return max(divs, key=lambda d: len(d.get_text(strip=True)))
    
    return soup


def extract_text(container):
    elements = container.find_all(['h1', 'h2', 'h3', 'p', 'li'])

    content = []
    for el in elements:
        c = el.get_text(strip=True)
        if c:
            content.append(c)

    return '\n'.join(content)

import time

data = []
count = 0

for _, row in df.iterrows():
    url = row['About Page']
    print(f"Scraping: {url}")

    soup = scrape_html(url)

    if not soup:
        data.append({
            "url": url,
            "about_content": None
        })
        continue

    container = find_content_container(soup)
    content = extract_text(container)

    data.append({
        "url": url,
        "about_content": content
    })
        
    time.sleep(1)
    
    if count == 5: break
    count += 1

pd.DataFrame(data).to_csv("scraped_data.csv", index=False)

Scraping: https://manifestationmatters.com/category/law-of-attraction/
Scraping: https://nextlevelsoul.com/blog/
Scraping: https://elliotoracle.com/
Scraping: nan
Scraping: nan
Scraping: https://conjuredcardea.com/
Scraping: nan
Scraping: nan
Scraping: https://goodinthehead.com/the-mental-hospital/
Scraping: https://www.enchanteddragonflyma.com/#AImbkY


In [ ]:

tags_per_link = []
count = 0
for index, row in df.iterrows():
    print(row["About Page"])
    tags = extract_tags(row["About Page"])
    if 'main' in tags:
        print("Has main")
    elif 'article' in tags:
        print("Has article")
    elif 'section' in tags:
        print("Has section")
    else:
        print('None')
    tags_per_link.append(tags)
    print(tags)

    if count == 20: break
    count += 1

for tag in tags_per_link:
    print(tag) 

In [20]:
df_about = pd.read_csv("scraped_data.csv")
df_about.head(5)

,url,about_content
0,https://manifestationmatters.com/category/law-...,Law of Attraction\nLaw of Attraction\nThis cat...
1,https://nextlevelsoul.com/blog/,403 Forbidden
2,https://elliotoracle.com/,Welcome to Elliot Oracle!\nFind daily wisdom w...
3,https://conjuredcardea.com/,"witch supplies, readings, magickal tips & prof..."
4,https://goodinthehead.com/the-mental-hospital/,My OLD Story\nBefore 2014 I spent 15 years in ...


In [23]:
'''
Products scraping
'''
df = pd.read_csv("../data/domains_annotated.csv")
url = df["Product Listing Page"].iloc[[0,2]]
print(url)

for u in url:
    soup = scrape_html(u)

    print("articles:", len(soup.find_all('article')))
    print("li:", len(soup.find_all('li')))
    print("div:", len(soup.find_all('div')))

0    https://manifestationmatters.com/manifestation...
2                       https://elliotoracle.com/shop/
Name: Product Listing Page, dtype: str
articles: 0
li: 26
div: 167
articles: 0
li: 74
div: 83


In [ ]:
blocks = soup.find_all('div')

for b in blocks[:26]:
    print(b.get_text(strip=True))
    print("-" * 50)

In [24]:
products = []

for block in blocks:
    title_tag = block.find(['h2','h3','a'])
    title = title_tag.get_text(strip=True) if title_tag else None

    price = None
    for t in block.stripped_strings:
        if '$' in t:
            price = t
            break

    link_tag = block.find('a', href=True)
    link = link_tag['href'] if link_tag else None

    if title:
        products.append({
            "title": title,
            "price": price,
            "link": link
        })

In [ ]:
for p in products:
    print(p)

In [27]:
def is_product(li):
    text = li.get_text(strip=True)

    return (
        len(text) > 20 and
        li.find('a') is not None
    )

In [29]:
url = df["Product Listing Page"].iloc[2]
soup = scrape_html(url)

In [30]:
blocks = soup.find_all('li')
filtered_blocks = [li for li in blocks if is_product(li)]

products = []
for li in filtered_blocks:
    title_tag = li.find(['h1', 'h2', 'h3', 'a'])
    title = title_tag.get_text(strip=True) if title_tag else None

    link_tag = li.find('a', href=True)
    link = link_tag['href'] if link_tag else None

    price = None
    for t in li.stripped_strings:
        if '$' in t:
            price = t
            break

    products.append({
        "title": title,
        "price": price,
        "link": link
    })

for p in products:
    print(p)

{'title': 'Shop', 'price': None, 'link': 'https://elliotoracle.com/shop/'}
{'title': 'Fearless Tarot | Signed Book', 'price': None, 'link': 'https://elliotoracle.com/shop/fearless-tarot-book-signed/'}
{'title': 'Tarot in Love | Signed Book', 'price': None, 'link': 'https://elliotoracle.com/shop/tarot-in-love-book-signed/'}
{'title': 'Gift Card (Electronic)', 'price': None, 'link': 'https://app.acuityscheduling.com/catalog.php?owner=18687323'}
{'title': 'Learn Tarot', 'price': None, 'link': 'https://elliotoracle.com/tarot-card-meanings/'}
{'title': 'Search Tarot Card Meanings', 'price': None, 'link': 'https://elliotoracle.com/tarot-card-meanings/'}
{'title': 'Minor Arcana', 'price': None, 'link': 'https://elliotoracle.com/tarot-card-meanings/minor-arcana/'}
{'title': 'Love and Relationships', 'price': None, 'link': 'https://elliotoracle.com/tarot-card-meanings/love-relationships/'}
{'title': 'Blog', 'price': None, 'link': 'https://elliotoracle.com/updates/'}
{'title': 'Beyond The Cards 